# Polymarket Dashboard

Explore active events and markets on [Polymarket](https://polymarket.com) using the [Gamma API](https://docs.polymarket.com/market-data/fetching-markets).

| Section | Description |
|---------|-------------|
| 1 | Top events by 24h volume |
| 2 | Top events by total volume |
| 3 | Top markets by 24h volume |
| 4 | Top markets by total volume |
| 5 | Biggest price movers (24h) |

In [ ]:
import requests
import pandas as pd
import json

GAMMA_API = "https://gamma-api.polymarket.com"

def fetch_events(order="volume24hr", limit=100):
    resp = requests.get(f"{GAMMA_API}/events", params={
        "active": "true",
        "closed": "false",
        "order": order,
        "ascending": "false",
        "limit": limit,
        "include_tag": "true",
    })
    resp.raise_for_status()
    return resp.json()

def fetch_markets(order="volume24hr", limit=100):
    resp = requests.get(f"{GAMMA_API}/markets", params={
        "active": "true",
        "closed": "false",
        "order": order,
        "ascending": "false",
        "limit": limit,
        "include_tag": "true",
    })
    resp.raise_for_status()
    return resp.json()

def events_df(events, n=10):
    rows = []
    for e in events[:n]:
        rows.append({
            "Title": e.get("title", "N/A"),
            "24h Volume": f"${float(e.get('volume24hr', 0)):,.0f}",
            "Total Volume": f"${float(e.get('volume', 0)):,.0f}",
            "Liquidity": f"${float(e.get('liquidity', 0)):,.0f}",
            "# Markets": len(e.get("markets", [])),
            "Slug": e.get("slug", ""),
        })
    df = pd.DataFrame(rows)
    df.index = range(1, len(df) + 1)
    df.index.name = "Rank"
    return df

def markets_df(markets, n=10):
    rows = []
    for m in markets[:n]:
        prices = m.get("outcomePrices", "[]") or "[]"
        rows.append({
            "Question": m.get("question", "N/A"),
            "24h Volume": f"${float(m.get('volume24hr', 0) or 0):,.0f}",
            "Total Volume": f"${float(m.get('volume', 0) or 0):,.0f}",
            "Liquidity": f"${float(m.get('liquidity', 0) or 0):,.0f}",
            "Outcome Prices": prices,
            "Slug": m.get("slug", ""),
        })
    df = pd.DataFrame(rows)
    df.index = range(1, len(df) + 1)
    df.index.name = "Rank"
    return df

def _get_tag_labels(item):
    return {t.get("label", "") for t in (item.get("tags") or [])}

def exclude_tags(items, tags):
    """Remove items that have any of the given tags."""
    tags = set(tags)
    return [i for i in items if not (_get_tag_labels(i) & tags)]

def keep_tags(items, tags):
    """Keep only items that have at least one of the given tags."""
    tags = set(tags)
    return [i for i in items if _get_tag_labels(i) & tags]

print("Helpers loaded.")

---
## Section 1: Top Events by 24h Volume

In [ ]:
events_by_24h = fetch_events(order="volume24hr")
print(f"Fetched {len(events_by_24h)} events (ordered by 24h volume)")
events_df(events_by_24h)

---
## Section 2: Top Events by Total Volume

In [ ]:
events_by_total = fetch_events(order="volume")
print(f"Fetched {len(events_by_total)} events (ordered by total volume)")
events_df(events_by_total)

---
## Section 3: Top Markets by 24h Volume

In [ ]:
markets_by_24h = fetch_markets(order="volume24hr")
print(f"Fetched {len(markets_by_24h)} markets (ordered by 24h volume)")
markets_df(markets_by_24h)

In [ ]:
print(json.dumps(events_by_24h[4], indent=2))

In [ ]:

print(json.dumps(markets_by_24h[0], indent=2))

---
## Section 4: Top Markets by Total Volume

In [ ]:
markets_by_total = fetch_markets(order="volumeNum")
print(f"Fetched {len(markets_by_total)} markets (ordered by total volume)")
markets_df(markets_by_total)

---
## Section 5: Biggest Price Movers (24h)

Uses `oneDayPriceChange` from the Gamma API to find markets with the largest absolute price movement in the past 24 hours.

In [ ]:
all_markets = fetch_markets(order="volume24hr", limit=1000)

# Only politics markets
politics = keep_tags(all_markets, ["Politics"])

# Everything except sports and crypto
filtered = exclude_tags(all_markets, ["Crypto Prices", "Sports"])

print(f"Kept {len(filtered)}/{len(all_markets)} markets (excluded {len(all_markets) - len(filtered)} tagged 'Crypto Prices')")

for m in filtered:
    m["_abs_change"] = abs(m.get("oneDayPriceChange") or 0)

movers = sorted(filtered, key=lambda m: m["_abs_change"], reverse=True)

rows = []
for m in movers[:20]:
    change = m.get("oneDayPriceChange") or 0
    last = m.get("lastTradePrice") or 0
    prices = m.get("outcomePrices", "[]") or "[]"
    rows.append({
        "Question": m.get("question", "N/A"),
        "24h Change": f"{change:+.1%}",
        "Last Price": f"{last:.2f}",
        "24h Volume": f"${float(m.get('volume24hr', 0) or 0):,.0f}",
        "Outcome Prices": prices,
        "Slug": m.get("slug", ""),
    })

df_movers = pd.DataFrame(rows)
df_movers.index = range(1, len(df_movers) + 1)
df_movers.index.name = "Rank"
print(f"Top 20 movers out of {len(all_markets)} active markets")
df_movers

In [ ]:
print(json.dumps(movers[7], indent=2))